In [56]:
# Imports and Setup
import os
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, balanced_accuracy_score
from collections import defaultdict, Counter

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

In [57]:
# Configuration
LATENT_DIM = 10
NUM_ROUNDS = 25
LOCAL_EPOCHS = 5
BATCH_SIZE = 32
LEARNING_RATE = 0.001
DP_NOISE_STD = 1e-5

print("Configuration:")
print(f"Latent dimension: {LATENT_DIM}")
print(f"Federated rounds: {NUM_ROUNDS}")
print(f"Local epochs: {LOCAL_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")

Configuration:
Latent dimension: 10
Federated rounds: 25
Local epochs: 5
Batch size: 32
Learning rate: 0.001


In [58]:
# Focal Loss Implementation
class FocalLoss(tf.keras.losses.Loss):
    """Focal Loss for handling class imbalance"""
    
    def __init__(self, alpha=0.25, gamma=2.0, name='focal_loss', **kwargs):
        super().__init__(name=name, **kwargs)
        self.alpha = alpha
        self.gamma = gamma
    
    def call(self, y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        
        y_true = tf.cast(y_true, tf.int32)
        y_true_one_hot = tf.one_hot(y_true, depth=tf.shape(y_pred)[-1])
        
        ce = -y_true_one_hot * tf.math.log(y_pred)
        
        p_t = tf.where(tf.equal(y_true_one_hot, 1), y_pred, 1 - y_pred)
        focal_weight = tf.pow(1 - p_t, self.gamma)
        
        alpha_t = tf.where(tf.equal(y_true_one_hot, 1), self.alpha, 1 - self.alpha)
        
        focal_loss = alpha_t * focal_weight * ce
        
        return tf.reduce_mean(tf.reduce_sum(focal_loss, axis=-1))
    
    def get_config(self):
        config = super().get_config()
        config.update({'alpha': self.alpha, 'gamma': self.gamma})
        return config
    
    @classmethod
    def from_config(cls, config):
        return cls(**config)


In [59]:
# Load Data
with open('../data/client_latent_features.pkl', 'rb') as f:
    client_latent_features = pickle.load(f)

with open('../data/client_data_processed.pkl', 'rb') as f:
    client_data_processed = pickle.load(f)

with open('../data/class_weights.pkl', 'rb') as f:
    class_weights = pickle.load(f)

NUM_CLIENTS = len(client_latent_features)
NUM_CLASSES = 2

print(f"\nLoaded data:")
print(f"Clients: {NUM_CLIENTS}")
print(f"Classes: {NUM_CLASSES}")

print(f"\nClass weights:")
for cls, weight in class_weights.items():
    class_name = "Healthy" if cls == 0 else "CKD"
    print(f"{class_name} ({cls}): {weight:.4f}")


Loaded data:
Clients: 4
Classes: 2

Class weights:
Healthy (0): 0.9457
CKD (1): 1.0746


In [60]:
# Prepare Client Data
clients = {}
total_class_dist = Counter()

for cid in range(NUM_CLIENTS):
    X_latent = client_latent_features[cid]
    y_labels = client_data_processed[cid][1].astype(np.int32)
    
    clients[cid] = {'X': X_latent, 'y': y_labels}
    total_class_dist.update(y_labels)

print("\nTraining data distribution:")
for cls, count in sorted(total_class_dist.items()):
    pct = 100 * count / sum(total_class_dist.values())
    class_name = "Healthy" if cls == 0 else "CKD"
    print(f"{class_name} (class {cls}): {count} samples ({pct:.1f}%)")



Training data distribution:
Healthy (class 0): 3100 samples (62.5%)
CKD (class 1): 1860 samples (37.5%)


In [61]:
# Create Validation Set
X_all_latent = np.vstack(client_latent_features)
y_all = np.concatenate([clients[cid]['y'] for cid in range(NUM_CLIENTS)])
X_all_latent, y_all = shuffle(X_all_latent, y_all, random_state=RANDOM_STATE)

n_val = max(1, int(0.15 * len(X_all_latent)))
X_val = X_all_latent[:n_val]
y_val = y_all[:n_val]

print(f"\nValidation set:")
print(f"Samples: {len(y_val)}")
print(f"Distribution: {dict(Counter(y_val))}")


Validation set:
Samples: 744
Distribution: {np.int32(0): 484, np.int32(1): 260}


In [62]:
# Build Classifier Model
def build_classifier(input_dim, num_classes):
    """Classifier with focal loss"""
    model = tf.keras.Sequential([
        layers.Input(shape=(input_dim,)),
        
        layers.Dense(128, kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.2),
        
        layers.Dense(64, kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.1),
        
        layers.Dense(32, kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=FocalLoss(alpha=0.75, gamma=2.0),
        metrics=['accuracy']
    )
    
    return model

In [63]:
# Federated Averaging Functions
def fed_avg_weighted(weights_list, weight_scalars):
    """Weighted federated averaging"""
    total = sum(weight_scalars)
    if total == 0:
        weight_scalars = [1.0] * len(weight_scalars)
        total = len(weight_scalars)
    
    n_layers = len(weights_list[0])
    averaged_weights = []
    
    for layer_idx in range(n_layers):
        layer_sum = None
        for model_weights, scalar in zip(weights_list, weight_scalars):
            layer_array = model_weights[layer_idx].astype(np.float64)
            scaled_layer = layer_array * (scalar / total)
            layer_sum = scaled_layer if layer_sum is None else layer_sum + scaled_layer
        averaged_weights.append(layer_sum.astype(np.float32))
    
    return averaged_weights

def add_noise_to_weights(weights, noise_std):
    """Add differential privacy noise"""
    return [w + np.random.normal(0, noise_std, w.shape).astype(np.float32) 
            for w in weights]

In [64]:
# Initialize Global Model
print("\nInitializing global model:")
global_model = build_classifier(LATENT_DIM, NUM_CLASSES)
global_weights = global_model.get_weights()
global_model.summary()


Initializing global model:


Model: "sequential_324"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1296 (Dense)              │ (None, 128)            │         1,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_972         │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_972 (Activation)     │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_648 (Dropout)           │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1297 (Dense)              │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_973         │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_973 (Activation)     │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_649 (Dropout)           │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1298 (Dense)              │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_974         │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_974 (Activation)     │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1299 (Dense)              │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,706 (49.63 KB)

 Trainable params: 12,258 (47.88 KB)

 Non-trainable params: 448 (1.75 KB)

In [65]:
# Federated Training Loop
best_f1 = 0.0
best_round = 0
best_weights = None
patience = 8
patience_counter = 0

metrics_history = {
    'train_accuracy': [],
    'val_accuracy': [],
    'val_loss': [],
    'val_f1': [],
    'val_recall': [],
    'val_precision': [],
    'weighted_score': []
}

print("\nStarting federated training:")

for round_num in range(1, NUM_ROUNDS + 1):
    print(f"\n--- Round {round_num}/{NUM_ROUNDS} ---")
    
    local_weights_list = []
    local_weight_scalars = []
    
    # Local training
    for cid in range(NUM_CLIENTS):
        X_client = clients[cid]['X']
        y_client = clients[cid]['y']
        
        if len(y_client) == 0:
            continue
        
        local_model = build_classifier(LATENT_DIM, NUM_CLASSES)
        local_model.set_weights(global_weights)
        
        local_model.fit(
            X_client, y_client,
            epochs=LOCAL_EPOCHS,
            batch_size=min(BATCH_SIZE, len(X_client)),
            verbose=0
        )
        
        local_weights = local_model.get_weights()
        noisy_weights = add_noise_to_weights(local_weights, DP_NOISE_STD)
        
        local_weights_list.append(noisy_weights)
        local_weight_scalars.append(len(y_client))
    
    # Aggregate
    if len(local_weights_list) > 0:
        global_weights = fed_avg_weighted(local_weights_list, local_weight_scalars)
        global_model.set_weights(global_weights)
    
    # Evaluate
    val_loss, val_accuracy = global_model.evaluate(X_val, y_val, verbose=0)
    y_val_pred = np.argmax(global_model.predict(X_val, verbose=0), axis=1)
    
    val_precision = precision_score(y_val, y_val_pred, pos_label=1, zero_division=0)
    val_recall = recall_score(y_val, y_val_pred, pos_label=1, zero_division=0)
    val_f1 = f1_score(y_val, y_val_pred, pos_label=1, zero_division=0)
    val_balanced_acc = balanced_accuracy_score(y_val, y_val_pred)
    
    weighted_score = 0.6 * val_recall + 0.4 * val_f1
    
    metrics_history['train_accuracy'].append(val_accuracy)
    metrics_history['val_accuracy'].append(val_accuracy)
    metrics_history['val_loss'].append(val_loss)
    metrics_history['val_f1'].append(val_f1)
    metrics_history['val_recall'].append(val_recall)
    metrics_history['val_precision'].append(val_precision)
    metrics_history['weighted_score'].append(weighted_score)
    
    print(f"Loss: {val_loss:.4f} | Accuracy: {val_accuracy:.4f}")
    print(f"Recall: {val_recall:.4f} | Precision: {val_precision:.4f} | F1: {val_f1:.4f}")
    print(f"Weighted Score: {weighted_score:.4f} (0.6*Recall + 0.4*F1)")
    
    # Early stopping
    if weighted_score > best_f1:
        best_f1 = weighted_score
        best_round = round_num
        best_weights = [w.copy() for w in global_weights]
        patience_counter = 0
        print(f"New best model!")
    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{patience})")
    
    if patience_counter >= patience:
        print(f"\nEarly stopping at round {round_num}")
        break

print(f"\nTraining complete - Best round: {best_round}")
print(f"Best weighted score: {best_f1:.4f}")



Starting federated training:

--- Round 1/25 ---
Loss: 0.2071 | Accuracy: 0.7446
Recall: 0.4846 | Precision: 0.6923 | F1: 0.5701
Weighted Score: 0.5188 (0.6*Recall + 0.4*F1)
New best model!

--- Round 2/25 ---
Loss: 0.1807 | Accuracy: 0.7742
Recall: 0.5231 | Precision: 0.7556 | F1: 0.6182
Weighted Score: 0.5611 (0.6*Recall + 0.4*F1)
New best model!

--- Round 3/25 ---
Loss: 0.1611 | Accuracy: 0.7863
Recall: 0.5731 | Precision: 0.7563 | F1: 0.6521
Weighted Score: 0.6047 (0.6*Recall + 0.4*F1)
New best model!

--- Round 4/25 ---
Loss: 0.1512 | Accuracy: 0.7836
Recall: 0.5308 | Precision: 0.7797 | F1: 0.6316
Weighted Score: 0.5711 (0.6*Recall + 0.4*F1)
No improvement (1/8)

--- Round 5/25 ---
Loss: 0.1492 | Accuracy: 0.7755
Recall: 0.5038 | Precision: 0.7751 | F1: 0.6107
Weighted Score: 0.5466 (0.6*Recall + 0.4*F1)
No improvement (2/8)

--- Round 6/25 ---
Loss: 0.1432 | Accuracy: 0.7796
Recall: 0.5231 | Precision: 0.7727 | F1: 0.6239
Weighted Score: 0.5634 (0.6*Recall + 0.4*F1)
No improve

In [66]:
# Save Models
OUTPUT_DIR = '../models'
os.makedirs(OUTPUT_DIR, exist_ok=True)

global_model.set_weights(best_weights)

global_model.save(f'{OUTPUT_DIR}/global_classifier_round{best_round}.keras')
global_model.save(f'{OUTPUT_DIR}/global_classifier.keras')

config = {
    'LATENT_DIM': LATENT_DIM,
    'NUM_ROUNDS': round_num,
    'NUM_CLASSES': NUM_CLASSES,
    'NUM_CLIENTS': NUM_CLIENTS,
    'BEST_ROUND': best_round,
    'BEST_WEIGHTED_SCORE': best_f1,
    'METRICS_HISTORY': metrics_history,
    'CLASS_WEIGHTS_USED': class_weights
}

with open(f'{OUTPUT_DIR}/training_config.pkl', 'wb') as f:
    pickle.dump(config, f)

print("\nSaved models and configuration")


Saved models and configuration
